# EDA - Global Cancer Dataset

Notebook de travail exploratoire (Jupyter) sur le dataset nettoye.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)

In [ ]:
DATA_PATH = Path('../../data/global_cancer_clean.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../../data/global_cancer_50years_extended_patient_level.csv')

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {DATA_PATH}')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(20)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols]
print('Numeric columns:', num_cols)
print('Categorical columns:', cat_cols)
df[num_cols].describe().T

In [ ]:
if 'Country' in df.columns:
    top_country = df['Country'].value_counts().head(15).sort_values()
    plt.figure(figsize=(10, 6))
    top_country.plot(kind='barh', color='#1f77b4')
    plt.title('Top pays (nombre de lignes)')
    plt.xlabel('Nombre de patients')
    plt.tight_layout()
    plt.show()

In [ ]:
if 'Year' in df.columns:
    yearly = df.groupby('Year').size().reset_index(name='Patients')
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=yearly, x='Year', y='Patients', marker='o', linewidth=1.5)
    plt.title('Evolution annuelle du nombre de cas')
    plt.tight_layout()
    plt.show()

In [ ]:
if {'Year', 'Country'}.issubset(df.columns):
    trend_country = (
        df.groupby(['Year', 'Country'])
        .size()
        .reset_index(name='Patients')
    )
    top5 = df['Country'].value_counts().head(5).index
    trend_top5 = trend_country[trend_country['Country'].isin(top5)]

    plt.figure(figsize=(12, 6))
    sns.lineplot(data=trend_top5, x='Year', y='Patients', hue='Country')
    plt.title('Evolution annuelle des cas - Top 5 pays')
    plt.tight_layout()
    plt.show()

In [ ]:
if 'Target_Severity_Score' in df.columns:
    plt.figure(figsize=(9, 5))
    sns.histplot(df['Target_Severity_Score'], bins=40, kde=True, color='#2ca02c')
    plt.title('Distribution de Target_Severity_Score')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_cols = [c for c in ['Age', 'Genetic_Risk', 'Air_Pollution', 'Alcohol_Use', 'Smoking', 'Obesity_Level', 'Survival_Years', 'Target_Severity_Score'] if c in df.columns]
if plot_cols:
    corr = df[plot_cols].corr(numeric_only=True)
    plt.figure(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
    plt.title('Matrice de correlation')
    plt.tight_layout()
    plt.show()

In [ ]:
if {'Country', 'Target_Severity_Score'}.issubset(df.columns):
    severity_country = df.groupby('Country', as_index=False)['Target_Severity_Score'].mean().sort_values('Target_Severity_Score', ascending=False)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=severity_country, x='Target_Severity_Score', y='Country', palette='viridis')
    plt.title('Score moyen de severite par pays')
    plt.tight_layout()
    plt.show()
    severity_country.head(10)